<a href="https://colab.research.google.com/github/giacomomolinari/liar-fake-news-detector/blob/main/models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Models

## Imports

In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from google.colab import drive
from pathlib import Path

from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 1. Getting the data

The notebook assumes that the LIAR dataset is available on your Google Drive at the path defined below. You can download the dataset from [William Yang Wang's website](https://sites.cs.ucsb.edu/~william/data/liar_dataset.zip).

In [70]:
DATASET_PATH = Path("/content/drive/MyDrive/datasets/liar_dataset/")

In [71]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
columns= ["ID", "Label", "Statement", "Subjects", "Speaker", "SpeakerJob", "State", "Party",
          "HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire", "Context"]

In [73]:
df_train = pd.read_csv(DATASET_PATH / "train.tsv", sep="\t", header=0, names=columns)
df_valid = pd.read_csv(DATASET_PATH / "valid.tsv", sep="\t", header=0, names=columns)
df_test = pd.read_csv(DATASET_PATH / "test.tsv", sep="\t", header=0, names=columns)

In [74]:
df_train.head()

,ID,Label,Statement,Subjects,Speaker,SpeakerJob,State,Party,HistBarelyTrue,HistFalse,HistHalfTrue,HistMostTrue,HistPantsFire,Context
0,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
1,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
2,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
3,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN
4,12465.json,true,The Chicago Bears have had more starting quart...,education,robin-vos,Wisconsin Assembly speaker,Wisconsin,republican,0.0,3.0,2.0,5.0,1.0,a an online opinion-piece


## 2. Preprocessing

### 2.1 Handling NaN values

As discussed in the eda notebook, only two columns have significant amounts of NaN values. I will remove these columns from the dataset to begin with.

In [75]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10239 entries, 0 to 10238
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              10239 non-null  object 
 1   Label           10239 non-null  object 
 2   Statement       10239 non-null  object 
 3   Subjects        10237 non-null  object 
 4   Speaker         10237 non-null  object 
 5   SpeakerJob      7341 non-null   object 
 6   State           8029 non-null   object 
 7   Party           10237 non-null  object 
 8   HistBarelyTrue  10237 non-null  float64
 9   HistFalse       10237 non-null  float64
 10  HistHalfTrue    10237 non-null  float64
 11  HistMostTrue    10237 non-null  float64
 12  HistPantsFire   10237 non-null  float64
 13  Context         10137 non-null  object 
dtypes: float64(5), object(9)
memory usage: 1.1+ MB


In [76]:
df_train_clean = df_train.drop(["SpeakerJob", "State"], axis=1)
df_train_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10239 entries, 0 to 10238
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              10239 non-null  object 
 1   Label           10239 non-null  object 
 2   Statement       10239 non-null  object 
 3   Subjects        10237 non-null  object 
 4   Speaker         10237 non-null  object 
 5   Party           10237 non-null  object 
 6   HistBarelyTrue  10237 non-null  float64
 7   HistFalse       10237 non-null  float64
 8   HistHalfTrue    10237 non-null  float64
 9   HistMostTrue    10237 non-null  float64
 10  HistPantsFire   10237 non-null  float64
 11  Context         10137 non-null  object 
dtypes: float64(5), object(7)
memory usage: 960.0+ KB


For `NaN` values in the `Context` column, we will replace them with "no context provided."

In [77]:
df_train_clean.loc[df_train_clean["Context"].isna(), "Context"] = "no context provided."

In [78]:
df_train_clean[df_train_clean.isna().any(axis=1)]

,ID,Label,Statement,Subjects,Speaker,Party,HistBarelyTrue,HistFalse,HistHalfTrue,HistMostTrue,HistPantsFire,Context
2141,638.json,false,The fact is that although we have had a presid...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no context provided.
9374,1626.json,false,"Joe, I keep hearing you every morning talking ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no context provided.


Only two rows with NaN values remain, and these have all values NaN except for the statement. I will drop these for simplicity.

In [79]:
df_train_clean = df_train_clean.dropna()

Let's define a convenience function to apply the same changes to the validation and test dataframes

In [80]:
def clean_dataset(df):
  df_clean = df.drop(["SpeakerJob", "State"], axis=1)
  df_clean.loc[df_clean["Context"].isna(), "Context"] = "no context provided."
  df_clean = df_clean.dropna()
  return df_clean

In [81]:
df_valid_clean = clean_dataset(df_valid)

df_valid_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1283 entries, 0 to 1282
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ID              1283 non-null   object
 1   Label           1283 non-null   object
 2   Statement       1283 non-null   object
 3   Subjects        1283 non-null   object
 4   Speaker         1283 non-null   object
 5   Party           1283 non-null   object
 6   HistBarelyTrue  1283 non-null   int64 
 7   HistFalse       1283 non-null   int64 
 8   HistHalfTrue    1283 non-null   int64 
 9   HistMostTrue    1283 non-null   int64 
 10  HistPantsFire   1283 non-null   int64 
 11  Context         1283 non-null   object
dtypes: int64(5), object(7)
memory usage: 120.4+ KB


### 2.2 Selecting and Aggregating Features

As discussed in the EDA notebook, we will aggregate the `pants-fire` label with the `false` label to ensure the model is only learning about statement truthfulness, rather than learning possibly normative features like whether a lie is obvious, brazen, shocking or extravagant.

In [82]:
df_train_reduced = df_train_clean.copy()

In [83]:
df_train_reduced.loc[df_train_reduced["Label"] == "pants-fire", "Label"] = "false"

Then we will drop the "speaker history" features which contain the count of entries with each of the labels for the statement's speaker, and thus introduce data leak risks.

In [84]:
df_train_reduced = df_train_reduced.drop(["HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire"], axis=1)

In [85]:
df_train_reduced.head()

,ID,Label,Statement,Subjects,Speaker,Party,Context
0,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,democrat,a floor speech.
1,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,democrat,Denver
2,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,none,a news release
3,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,democrat,an interview on CNN
4,12465.json,true,The Chicago Bears have had more starting quart...,education,robin-vos,republican,a an online opinion-piece


Again we define a utility function to perform this transformation.

In [86]:
def aggregate_and_drop(df):
  df_res = df.copy()
  df_res.loc[df_res["Label"] == "pants-fire", "Label"] = "false"
  df_res = df_res.drop(["HistBarelyTrue", "HistFalse", "HistHalfTrue", "HistMostTrue", "HistPantsFire"], axis=1)
  return df_res

In [87]:
df_valid_reduced = aggregate_and_drop(df_valid_clean)

df_valid_reduced.head()

,ID,Label,Statement,Subjects,Speaker,Party,Context
0,238.json,false,"When Obama was sworn into office, he DID NOT u...","obama-birth-certificate,religion",chain-email,none,no context provided.
1,7891.json,false,Says Having organizations parading as being so...,"campaign-finance,congress,taxes",earl-blumenauer,democrat,a U.S. Ways and Means hearing
2,8169.json,half-true,Says nearly half of Oregons children are poor.,poverty,jim-francesconi,none,an opinion article
3,929.json,half-true,On attacks by Republicans that various program...,"economy,stimulus",barack-obama,democrat,interview with CBS News
4,9416.json,false,Says when armed civilians stop mass shootings ...,guns,jim-rubens,republican,"in an interview at gun shop in Hudson, N.H."


### 2.3 Multi-hot encoding of Subject feature

The Subject feature is a list of tags, so we will encode it by creating a new feature for each (sufficiently common) tag, which is 1 iff the sample in question has that subject tag.

In [88]:
subject_set = set([])
subject_series = []
for subject in df_train_reduced["Subjects"]:
  if not isinstance(subject, str):
    print(subject)
  subject = subject.replace(", ", ",")
  split = subject.split(",")
  for s in split:
    if s not in subject_set:
      subject_set.add(s)
    subject_series.append(s)

In [89]:
subject_series = pd.Series(subject_series)
counts = subject_series.value_counts()
counts.count(), counts[counts > 40].count()

(np.int64(142), np.int64(100))

In [90]:
common_subjects = set(counts[counts > 40].index)

there are 100 subjects which appear over 40 times, while all others appear less than 40 times each. I will aggregate these infrequent subject tags into a new tag "other"

In [91]:
df_train_listsubject = df_train_reduced.copy()
df_train_listsubject["Subjects"] = (df_train_listsubject["Subjects"].apply(lambda x: x.replace(", ", ",").split(","))
  .apply(lambda x: [s if s in common_subjects else "other" for s in x])
  .apply(lambda x: list(set(x)))) # remove duplicates

In [92]:
df_train_listsubject.shape

(10237, 7)

In [93]:
df_exploded = df_train_listsubject.explode("Subjects")
df_exploded.head()

,ID,Label,Statement,Subjects,Speaker,Party,Context
0,10540.json,half-true,When did the decline of coal start? It started...,job-accomplishments,scott-surovell,democrat,a floor speech.
0,10540.json,half-true,When did the decline of coal start? It started...,energy,scott-surovell,democrat,a floor speech.
0,10540.json,half-true,When did the decline of coal start? It started...,history,scott-surovell,democrat,a floor speech.
1,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,democrat,Denver
2,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,none,a news release


In [94]:
one_hot = pd.get_dummies(df_exploded['Subjects'])

In [95]:
one_hot.shape

(22158, 101)

In [96]:
multi_hot = one_hot.groupby(one_hot.index).sum()
multi_hot.shape

(10237, 101)

In [97]:
df_train_multihot = df_train_reduced.merge(multi_hot, left_index=True, right_index=True).drop("Subjects", axis=1)
df_train_multihot.shape


(10237, 107)

To the original 7 features we added 101 subject features (100 for common subjects, 1 for "other"), and then removed the pre-existing "Subjects" feature, for a total of 107 features in the final dataset.

Again we define a utility function to more easily perform this transformation on the validation and test datasets.

In [98]:
def multi_hot_encode(df, subject_set):
  df_listsubject = df.copy()
  df_listsubject["Subjects"] = (df_listsubject["Subjects"].apply(lambda x: x.replace(", ", ",").split(","))
    .apply(lambda x: [s if s in subject_set else "other" for s in x])
    .apply(lambda x: list(set(x)))) # remove duplicates

  df_exploded = df_listsubject.explode("Subjects") # add new rows for each value in "Subjects" list, copying other fields

  one_hot = pd.get_dummies(df_exploded['Subjects']) # one-hot encode all possible "Subject" values
  multi_hot = one_hot.groupby(one_hot.index).sum()  # group back by index to restore origina number of columns
  return multi_hot.merge(df_listsubject.drop("Subjects", axis=1), left_index=True, right_index=True)

In [123]:
df_valid_reduced.shape

(1283, 7)

In [122]:
df_valid_multihot = multi_hot_encode(df_valid_reduced, common_subjects)
df_valid_multihot.shape

(1283, 107)

### 2.4 Statement Length

I will remove all statements with over 40 words from the dataset. As shown in the EDA notebook, this adds up to about 100 statements, some of which are hundreds of words long, well above the dataset mode of 15. By removing these long statements we reduce the amount of padding needed to ensure all sequences processed by the RNN have the same length, which can improve performance.

In [99]:
df_train_preprocessed = df_train_multihot.copy()
df_train_preprocessed["StatementLength"] = df_train_preprocessed["Statement"].apply(lambda x: len(x.split(" ")))
df_train_preprocessed[["Statement", "StatementLength"]].head()

,Statement,StatementLength
0,When did the decline of coal start? It started...,24
1,"Hillary Clinton agrees with John McCain ""by vo...",19
2,Health care reform legislation is likely to ma...,12
3,The economic turnaround started at the end of ...,10
4,The Chicago Bears have had more starting quart...,27


In [100]:
df_train_preprocessed.shape

(10237, 108)

In [101]:
df_train_preprocessed = df_train_preprocessed[df_train_preprocessed["StatementLength"] < 40]
df_train_preprocessed.shape

(10111, 108)

As usual we define a utility function for this transformation.

In [102]:
def drop_statements_longer_than(df, maxLength):
  df_res = df.copy()
  df_res["StatementLength"] = df_res["Statement"].apply(lambda x: len(x.split(" ")))
  df_res = df_res[df_res["StatementLength"] < maxLength]
  return df_res

In [124]:
df_valid_preprocessed = drop_statements_longer_than(df_valid_multihot, 40)
df_valid_preprocessed.shape

(1269, 108)

## 3. Baseline Model 1: Metadata-only classifier

As a first baseline let's build a model that tries to classify statements based only on the available metadata: the name of the speaker, their political affiliation, and the topics touched on by the statement. I will also include the statement length, which is a derived piece of metadata and may potentially be helpful.

### 3.1 Preparing the data

We need a little bit more preprocessing before the metadata is ready to be used by a ML classifier. In particular, the remaining categorical features (`Speaker` and `Party`) must be converted to numerical ones, and the numerical feature `StatementLength` must be normalized.

#### 3.1.1 Normalizing StatementLength

In [103]:
subjects_set_list = list(common_subjects.union({"other"}))
meta_features = ["Speaker", "Party", "StatementLength"] + subjects_set_list

X_train_meta = df_train_preprocessed[meta_features]
y_train_meta = df_train_preprocessed["Label"]

X_train_meta.head()

,Speaker,Party,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,...,campaign-advertising,agriculture,diversity,candidates-biography,energy,deficit,housing,jobs,animals,veterans
0,scott-surovell,democrat,24,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,barack-obama,democrat,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,blog-posting,none,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,charlie-crist,democrat,10,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,robin-vos,republican,27,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [104]:
min_max_scaler = MinMaxScaler(feature_range=(0, 1))
X_train_meta[["StatementLength"]] = min_max_scaler.fit_transform(X_train_meta[["StatementLength"]])
X_train_meta.head()

/tmp/ipykernel_1954/501206241.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_meta[["StatementLength"]] = min_max_scaler.fit_transform(X_train_meta[["StatementLength"]])


,Speaker,Party,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,...,campaign-advertising,agriculture,diversity,candidates-biography,energy,deficit,housing,jobs,animals,veterans
0,scott-surovell,democrat,0.594595,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,barack-obama,democrat,0.459459,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,blog-posting,none,0.270270,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,charlie-crist,democrat,0.216216,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,robin-vos,republican,0.675676,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [105]:
statement_length_normalized = X_train_meta["StatementLength"]
statement_length_normalized.describe()

,StatementLength
count,10111.000000
mean,0.421080
std,0.194204
min,0.000000
25%,0.270270
50%,0.405405
75%,0.540541
max,1.000000


#### 3.1.2. Encoding Speaker

In [106]:
speaker_counts = X_train_meta["Speaker"].value_counts()
speaker_counts

,count
Speaker,
barack-obama,483
donald-trump,269
hillary-clinton,233
mitt-romney,175
john-mccain,148
...,...
jalen-ross,1
amardeep-kaleka,1
jennie-lou-leeder,1


Clearly there are too many speakers for one-hot encoding. But for many of these speakers, they appear too infrequently for the model to actually be able to learn much about them. So let's look at how many speakers appear frequently.

In [107]:
speaker_counts[speaker_counts>20].count()

np.int64(65)

Interestingly, only 65 speakers appear over 20 times. So we will aggregate all other speakers to a new value "other", and then one-hot encode this feature to generate 66 more feature (removing the existing `Speaker` feature)

In [108]:
common_speakers = set(speaker_counts[speaker_counts>20].index)

X_train_speaker_encoded = X_train_meta.copy()
X_train_speaker_encoded["Speaker"] = X_train_meta["Speaker"].apply(lambda x: x if x in common_speakers else "other")
X_train_speaker_encoded = pd.get_dummies(X_train_speaker_encoded, columns=["Speaker"])
X_train_speaker_encoded.head()

,Party,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,infrastructure,...,Speaker_rush-limbaugh,Speaker_sarah-palin,Speaker_scott-walker,Speaker_sherrod-brown,Speaker_tammy-baldwin,Speaker_ted-cruz,Speaker_terry-mcauliffe,Speaker_tim-kaine,Speaker_tom-barrett,Speaker_tommy-thompson
0,democrat,0.594595,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,democrat,0.459459,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,none,0.270270,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,democrat,0.216216,0,0,1,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,republican,0.675676,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False


#### 3.1.3 Encoding Party

We can one-hot encode the `Party` feature in a similar way.

In [109]:
party_counts = X_train_meta["Party"].value_counts()
party_counts

,count
Party,
republican,4451
democrat,3281
none,1726
organization,216
independent,146
newsmaker,55
libertarian,39
journalist,38
activist,38


Again it makes sense to aggregate parties that are too infrequent to be actually learned by the model.

In [110]:
common_parties = set(party_counts[party_counts>20].index)
X_train_party_encoded = X_train_speaker_encoded.copy()

X_train_party_encoded["Party"] = X_train_speaker_encoded["Party"].apply(lambda x: x if x in common_parties else "other")
X_train_party_encoded = pd.get_dummies(X_train_party_encoded, columns=["Party"])
X_train_party_encoded.head()

,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,infrastructure,government-efficiency,...,Party_democrat,Party_independent,Party_journalist,Party_libertarian,Party_newsmaker,Party_none,Party_organization,Party_other,Party_republican,Party_talk-show-host
0,0.594595,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
1,0.459459,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
2,0.270270,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
3,0.216216,0,0,1,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
4,0.675676,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False


In [111]:
X_train_party_encoded.shape

(10111, 180)

Finally we convert the boolean values to integers for numerical processing and consistency.

In [134]:
X_train_meta = X_train_party_encoded.copy()

X_statement_length = X_train_meta["StatementLength"].copy()
X_train_meta = X_train_meta.map(lambda x: int(x))
X_train_meta["StatementLength"] = X_statement_length

X_train_meta.head()

,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,infrastructure,government-efficiency,...,Party_democrat,Party_independent,Party_journalist,Party_libertarian,Party_newsmaker,Party_none,Party_organization,Party_other,Party_republican,Party_talk-show-host
0,0.594595,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0.459459,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0.270270,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0.216216,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0.675676,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


As before we define a function that performs all these transformations on a dataset.

In [135]:
def meta_preprocessing(df):
  df_res = df.copy()

  min_max_scaler = MinMaxScaler(feature_range=(0, 1))
  df_res["StatementLength"] = min_max_scaler.fit_transform(df[["StatementLength"]])

  df_res["Speaker"] = df_res["Speaker"].apply(lambda x: x if x in common_speakers else "other")
  df_res = pd.get_dummies(df_res, columns=["Speaker"])

  df_res["Party"] = df_res["Party"].apply(lambda x: x if x in common_parties else "other")
  df_res = pd.get_dummies(df_res, columns=["Party"])

  df_statement_length = df_res["StatementLength"].copy()
  df_res = df_res.map(lambda x: int(x))
  df_res["StatementLength"] = df_statement_length

  return df_res

In [136]:
X_valid_meta = df_valid_preprocessed[meta_features]
y_valid_meta = df_valid_preprocessed["Label"]

X_valid_meta.head()

,Speaker,Party,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,...,campaign-advertising,agriculture,diversity,candidates-biography,energy,deficit,housing,jobs,animals,veterans
0,chain-email,none,26,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,earl-blumenauer,democrat,32,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,jim-francesconi,none,8,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,barack-obama,democrat,33,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,jim-rubens,republican,22,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [140]:
X_valid_meta = meta_preprocessing(X_valid_meta)
X_valid_meta.head()

,StatementLength,city-government,families,economy,stimulus,marriage,message-machine-2012,state-budget,infrastructure,government-efficiency,...,Party_democrat,Party_independent,Party_journalist,Party_libertarian,Party_newsmaker,Party_none,Party_organization,Party_other,Party_republican,Party_talk-show-host
0,0.638889,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,0.805556,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0.138889,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0.833333,0,0,1,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0.527778,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


This has one fewer column than our preprocessed training dataset, so there is probably a categorical feature that never appears in the validation set but which appears in the training set.

In [142]:
X_valid_meta.columns.difference(X_train_meta.columns)
X_train_meta.columns.difference(X_valid_meta.columns)


Index(['Speaker_rush-limbaugh'], dtype='object')

It looks like a speaker that appears frequently in training data never appears in the validation data. We can fix this by just adding this column to the validation dataset and filling it with zeros

In [143]:
X_valid_meta["Speaker_rush-limbaugh"] = 0
X_valid_meta.shape

(1269, 180)

### 3.2 Selecting and training models

#### 3.2.1 Defining the evaluation measures

We will evaluate our models by first converting the labels into integers and then calculating the MAE. This preserves the intuitive ordering that is intrinsic in these truthfulness labels. For example, if a statement is almost-true, it seems better to have classified it as true than to have classified it as false (although one could plausibly wish to specify just how much better that is, I will for simplicity just take this to be specified by the MAE metric).

I will also evaluate our model using the usual accuracy metric, for comparison and more interpretable results.

In [144]:
label_to_int = {"false": 0, "barely-true": 1, "half-true": 2, "mostly-true": 3, "true": 4}

y_train_meta_int = y_train_meta.apply(lambda x: label_to_int[x])
y_valid_meta_int = y_valid_meta.apply(lambda x: label_to_int[x])

In [145]:
scoring_rule = "accuracy"
scoring_rule_ordinal = "neg_mean_absolute_error"

#### 3.2.2 Simple Baseline Classifiers

As a sanity check, it's useful to define a **Majority Class Classifier**. This is a classifier which just predicts the most frequent class regardless of input. Its performance on the task provides a performance floor which our models should be able to clear if they have learned anything at all

In [115]:
y_mode = y_train_meta_int.mode()[0]
y_mode

np.int64(0)

In [150]:
def majority_class_classifier(X, y):
  return np.ones((len(X), 1))* y.mode()[0]

In [151]:
mean_absolute_error(y_valid_meta_int, majority_class_classifier(X_valid_meta, y_train_meta_int))

1.6800630417651694

In [152]:
accuracy_score(y_valid_meta_int, majority_class_classifier(X_valid_meta, y_train_meta_int))

0.2978723404255319

As expected this has an accuracy of around 27%, as around 27% of statements in the training dataset are false.

#### 3.2.3 Tree classifier

In [119]:
tree_model = DecisionTreeClassifier(random_state=42)

tree_model

DecisionTreeClassifier(random_state=42)

In [120]:
tree_model_cv = -cross_val_score(tree_model, X_train_meta, y_train_meta_int, scoring=scoring_rule_ordinal, cv=10)

tree_model_cv.mean()

np.float64(1.4898591775059329)

#### 3.2.4 Random forest classifier

In [121]:
forest_model = RandomForestClassifier(random_state=42)

forest_model_cv = cross_val_score(forest_model, X_train_meta, y_train_meta_int, scoring=scoring_rule, cv=5)

forest_model_cv

array([0.24320316, 0.27101879, 0.26013848, 0.26261128, 0.26706231])

## 4. Baseline Model 2: Text-only classifier

## 5. Main Model: Text and metadata classifier